# 03 · Modelagem e carga: camada Gold (Esquema Estrela)

**Objetivo da etapa:** montar o modelo dimensional que responde às perguntas de negócio.

**Grão da tabela fato:** *1 linha por pedido* (99.441 pedidos, com todos os status). As análises
de prazo filtram `flag_entregue_valido = true`.

```
                   dim_tempo  (papéis: compra · prometida · entrega)
                       │
  dim_cliente ──── fato_pedido ──── dim_vendedor
                       │
                   dim_produto
```

**Decisões de modelagem (regras de negócio):**

| Regra | Descrição |
|---|---|
| **G01 · Vendedor e produto principais** | Um pedido pode ter vários itens e vendedores. O pedido é atribuído ao item de **maior preço**; em caso de empate, ao de menor `item_seq`. A quantidade de vendedores fica registrada em `qtd_vendedores`. |
| **G02 · Membro desconhecido** | Os 775 pedidos sem itens (cancelados ou indisponíveis) apontam para o membro `sk = -1` das dimensões, preservando a integridade referencial. |
| **G03 · Atraso** | A promessa é uma *data*, sem horário. Por isso `dias_atraso = data(entrega) − data prometida`, e `flag_atraso = dias_atraso > 0`. |
| **G04 · Postagem atrasada (responsabilidade do vendedor)** | `dt_postagem > maior dt_limite_postagem` dos itens do pedido. |
| **G05 · Distância** | Distância em linha reta (fórmula de Haversine) entre os centroides dos CEPs do cliente e do vendedor principal. |
| **G06 · Durações** | São calculadas em dias decimais e ficam nulas quando o pedido tem inconsistência temporal (`dq_inconsistente`). |
| **G07 · Região** | Cada UF é mapeada para a sua macrorregião do IBGE. |
| **G08 · Chaves substitutas** | Inteiros sequenciais atribuídos por ordenação da chave natural, o que é determinístico. Na `dim_tempo`, a chave é a data no formato `aaaammdd`. |
| **G09 · Pagamento** | `valor_pago` é a soma de todos os pagamentos do pedido; `tipo_pagamento` é o tipo de maior valor; `parcelas` é o máximo de parcelas. |

### Configuração do ambiente
Esta célula prepara o ambiente de execução. No Google Colab, ela:
1. clona o repositório do projeto, que contém o módulo `scripts/common.py`;
2. instala o PySpark e o Delta Lake;
3. monta o Google Drive, onde o lakehouse fica persistido.

Na primeira execução, o Colab pede autorização de acesso ao Drive.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/FilippeDumont/mvp-olist-atrasos-entrega.git"
REPO_DIR = "/content/repo"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    # O Spark precisa de Java: instala um JDK apenas se a máquina não tiver um
    if subprocess.run(["bash", "-c", "command -v java"], capture_output=True).returncode != 0:
        subprocess.run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub"], check=True)
    sys.path.insert(0, f"{REPO_DIR}/scripts")
else:  # execução local de teste
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import common as C

C.mount_drive()
spark = C.get_spark()
print("Spark", spark.version, "| Delta Lake habilitado")

In [ ]:
from pyspark.sql import functions as F, Window as W

C.banner("03 · Modelagem e carga (Gold)")
s = {t: C.read_table("silver", t) for t in
     ["pedidos", "clientes", "vendedores", "itens_pedido", "pagamentos", "avaliacoes",
      "produtos", "geolocalizacao_cep"]}

REGIAO = {"AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
          "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste", "PE": "Nordeste",
          "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
          "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
          "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
          "PR": "Sul", "RS": "Sul", "SC": "Sul"}
mapa_regiao = F.create_map(*[F.lit(x) for kv in REGIAO.items() for x in kv])

def com_sk(df, chave, sk):
    """G08: chave substituta determinística."""
    return df.withColumn(sk, F.row_number().over(W.orderBy(chave)).cast("int"))

def membro_desconhecido(df, sk):
    """G02: linha sk=-1 com atributos nulos/'desconhecido'."""
    vals = []
    for f in df.schema.fields:
        if f.name == sk:
            vals.append(F.lit(-1).cast(f.dataType).alias(f.name))
        elif f.dataType.simpleString() == "string":
            vals.append(F.lit("desconhecido").alias(f.name))
        else:
            vals.append(F.lit(None).cast(f.dataType).alias(f.name))
    return df.limit(1).select(*vals).unionByName(df)

geo = s["geolocalizacao_cep"].select("cep_prefixo", "lat", "lng")

## Dimensões

In [ ]:
# dim_tempo: calendário contínuo, cobrindo todas as datas do modelo
# (limites calculados como DATE dentro do Spark, sem passar pelo Python, para evitar conversão de fuso)
limites = s["pedidos"].select(
    F.least(F.to_date(F.min("dt_compra")), F.min("dt_entrega_prometida")).alias("ini"),
    F.greatest(F.to_date(F.max("dt_entrega")), F.max("dt_entrega_prometida")).alias("fim"))
NOMES_MES = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho", "agosto",
             "setembro", "outubro", "novembro", "dezembro"]
NOMES_DIA = ["domingo", "segunda", "terça", "quarta", "quinta", "sexta", "sábado"]
dim_tempo = (limites.select(F.explode(F.sequence("ini", "fim")).alias("data"))
    .select(F.date_format("data", "yyyyMMdd").cast("int").alias("sk_data"), "data",
            F.year("data").alias("ano"), F.quarter("data").alias("trimestre"), F.month("data").alias("mes"),
            F.element_at(F.array(*[F.lit(m) for m in NOMES_MES]), F.month("data")).alias("nome_mes"),
            F.date_format("data", "yyyy-MM").alias("ano_mes"), F.dayofmonth("data").alias("dia"),
            F.dayofweek("data").alias("dia_semana"),
            F.element_at(F.array(*[F.lit(d) for d in NOMES_DIA]), F.dayofweek("data")).alias("nome_dia_semana"),
            F.dayofweek("data").isin(1, 7).alias("fim_de_semana")))
dim_tempo = membro_desconhecido(dim_tempo, "sk_data")

dim_cliente = com_sk(
    s["clientes"].join(geo, "cep_prefixo", "left")
    .withColumn("regiao", mapa_regiao[F.col("uf")]), "cliente_id", "sk_cliente"
).select("sk_cliente", "cliente_id", "cliente_unico_id", "cep_prefixo", "cidade", "uf", "regiao", "lat", "lng")

dim_vendedor = membro_desconhecido(com_sk(
    s["vendedores"].join(geo, "cep_prefixo", "left")
    .withColumn("regiao", mapa_regiao[F.col("uf")]), "vendedor_id", "sk_vendedor"
).select("sk_vendedor", "vendedor_id", "cep_prefixo", "cidade", "uf", "regiao", "lat", "lng"), "sk_vendedor")

dim_produto = membro_desconhecido(com_sk(
    s["produtos"].withColumn("volume_cm3", F.col("comprimento_cm") * F.col("altura_cm") * F.col("largura_cm")),
    "produto_id", "sk_produto"
).select("sk_produto", "produto_id", "categoria", "categoria_en", "peso_g", "comprimento_cm", "altura_cm",
         "largura_cm", "volume_cm3", "qtd_fotos", "qtd_caracteres_nome", "qtd_caracteres_descricao"), "sk_produto")

## Tabela fato

In [ ]:
itens = s["itens_pedido"]
# G01: item principal = maior preço (desempate: menor item_seq)
w_item = W.partitionBy("pedido_id").orderBy(F.col("preco").desc(), F.col("item_seq"))
principal = (itens.withColumn("_rk", F.row_number().over(w_item)).filter("_rk = 1")
             .select("pedido_id", F.col("produto_id").alias("produto_principal_id"),
                     F.col("vendedor_id").alias("vendedor_principal_id")))
agg_itens = itens.groupBy("pedido_id").agg(
    F.count("*").alias("qtd_itens"), F.countDistinct("vendedor_id").alias("qtd_vendedores"),
    F.round(F.sum("preco"), 2).alias("valor_produtos"), F.round(F.sum("frete"), 2).alias("valor_frete"),
    F.max("dt_limite_postagem").alias("dt_limite_postagem"))

# G09: pagamentos
w_pg = W.partitionBy("pedido_id").orderBy(F.col("valor").desc(), F.col("pagamento_seq"))
pg = s["pagamentos"]
agg_pg = (pg.groupBy("pedido_id").agg(F.round(F.sum("valor"), 2).alias("valor_pago"),
                                      F.max("parcelas").alias("parcelas"))
          .join(pg.withColumn("_rk", F.row_number().over(w_pg)).filter("_rk = 1")
                  .select("pedido_id", "tipo_pagamento"), "pedido_id"))

av = s["avaliacoes"].select("pedido_id", F.col("nota").alias("nota_avaliacao"), "tem_comentario")

dias = lambda a, b: F.round((F.col(b).cast("long") - F.col(a).cast("long")) / 86400.0, 2)

base = (s["pedidos"]
    .join(agg_itens, "pedido_id", "left").join(principal, "pedido_id", "left")
    .join(agg_pg, "pedido_id", "left").join(av, "pedido_id", "left")
    .join(dim_cliente.select("sk_cliente", "cliente_id", F.col("lat").alias("lat_c"), F.col("lng").alias("lng_c"),
                             F.col("uf").alias("uf_c")), "cliente_id", "left")
    .join(dim_vendedor.filter("sk_vendedor > 0").select(
              "sk_vendedor", F.col("vendedor_id").alias("vendedor_principal_id"),
              F.col("lat").alias("lat_v"), F.col("lng").alias("lng_v"), F.col("uf").alias("uf_v")),
          "vendedor_principal_id", "left")
    .join(dim_produto.filter("sk_produto > 0").select("sk_produto", F.col("produto_id").alias("produto_principal_id")),
          "produto_principal_id", "left"))

valido = (F.col("status") == "delivered") & F.col("dt_entrega").isNotNull() & ~F.col("dq_inconsistente")
so_valido = lambda expr: F.when(~F.col("dq_inconsistente"), expr)   # G06

# G05: Haversine (km)
rad = F.radians
hav = (F.pow(F.sin((rad("lat_c") - rad("lat_v")) / 2), 2)
       + F.cos(rad("lat_v")) * F.cos(rad("lat_c")) * F.pow(F.sin((rad("lng_c") - rad("lng_v")) / 2), 2))
distancia = F.round(2 * 6371.0 * F.asin(F.sqrt(hav)), 1)

sk_data = lambda c: F.coalesce(F.date_format(c, "yyyyMMdd").cast("int"), F.lit(-1))

fato_pedido = base.select(
    "pedido_id",
    F.coalesce("sk_cliente", F.lit(-1)).alias("sk_cliente"),
    F.coalesce("sk_vendedor", F.lit(-1)).alias("sk_vendedor"),
    F.coalesce("sk_produto", F.lit(-1)).alias("sk_produto"),
    sk_data("dt_compra").alias("sk_data_compra"),
    sk_data("dt_entrega_prometida").alias("sk_data_prometida"),
    sk_data("dt_entrega").alias("sk_data_entrega"),
    "status",
    F.coalesce("qtd_itens", F.lit(0)).cast("int").alias("qtd_itens"),
    F.coalesce("qtd_vendedores", F.lit(0)).cast("int").alias("qtd_vendedores"),
    "valor_produtos", "valor_frete", "valor_pago", "tipo_pagamento", "parcelas",
    so_valido(F.round(dias("dt_compra", "dt_aprovacao") * 24, 2)).alias("horas_aprovacao"),
    so_valido(dias("dt_compra", "dt_postagem")).alias("dias_ate_postagem"),
    so_valido(dias("dt_postagem", "dt_entrega")).alias("dias_transporte"),
    so_valido(dias("dt_compra", "dt_entrega")).alias("dias_entrega"),
    F.datediff("dt_entrega_prometida", F.to_date("dt_compra")).alias("dias_prometidos"),
    F.when(valido, F.datediff(F.to_date("dt_entrega"), "dt_entrega_prometida")).alias("dias_atraso"),   # G03
    F.when(valido, F.datediff(F.to_date("dt_entrega"), "dt_entrega_prometida") > 0).alias("flag_atraso"),
    F.when(~F.col("dq_inconsistente") & F.col("dt_postagem").isNotNull(),
           F.col("dt_postagem") > F.col("dt_limite_postagem")).alias("flag_postagem_atrasada"),       # G04
    distancia.alias("distancia_km"),
    (F.col("uf_c") == F.col("uf_v")).alias("mesma_uf"),
    "nota_avaliacao", "tem_comentario",
    valido.alias("flag_entregue_valido"),
    F.col("dq_inconsistente").alias("flag_inconsistencia_temporal"))

## Gravação na camada Gold

In [ ]:
gold = {"dim_tempo": dim_tempo, "dim_cliente": dim_cliente, "dim_vendedor": dim_vendedor,
        "dim_produto": dim_produto, "fato_pedido": fato_pedido}
for nome, df in gold.items():
    with C.Stage("03_carga_gold", f"gold.{nome}") as st:
        st.rows = C.write_table(df, "gold", nome)

## Verificações de carga
1. **Unicidade do grão:** o número de linhas da fato deve ser igual ao número de pedidos da Silver.
2. **Integridade referencial:** toda chave estrangeira da fato deve existir na dimensão correspondente.

In [ ]:
C.register_views("gold", list(gold))
C.register_views("silver", ["pedidos"])
checks = spark.sql("""
SELECT 'linhas fato = pedidos silver' AS verificacao,
       (SELECT count(*) FROM gold_fato_pedido) = (SELECT count(*) FROM silver_pedidos) AS ok
UNION ALL SELECT 'pedido_id único na fato',
       (SELECT count(*) - count(DISTINCT pedido_id) FROM gold_fato_pedido) = 0
UNION ALL SELECT 'FK cliente íntegra',  (SELECT count(*) FROM gold_fato_pedido f LEFT ANTI JOIN gold_dim_cliente d USING (sk_cliente)) = 0
UNION ALL SELECT 'FK vendedor íntegra', (SELECT count(*) FROM gold_fato_pedido f LEFT ANTI JOIN gold_dim_vendedor d USING (sk_vendedor)) = 0
UNION ALL SELECT 'FK produto íntegra',  (SELECT count(*) FROM gold_fato_pedido f LEFT ANTI JOIN gold_dim_produto d USING (sk_produto)) = 0
UNION ALL SELECT 'FK data compra íntegra',    (SELECT count(*) FROM gold_fato_pedido f LEFT ANTI JOIN gold_dim_tempo d ON f.sk_data_compra = d.sk_data) = 0
UNION ALL SELECT 'FK data prometida íntegra', (SELECT count(*) FROM gold_fato_pedido f LEFT ANTI JOIN gold_dim_tempo d ON f.sk_data_prometida = d.sk_data) = 0
UNION ALL SELECT 'FK data entrega íntegra',   (SELECT count(*) FROM gold_fato_pedido f LEFT ANTI JOIN gold_dim_tempo d ON f.sk_data_entrega = d.sk_data) = 0
""")
checks.show(truncate=False)
assert all(r.ok for r in checks.collect()), "Falha nas verificações de carga"

In [ ]:
spark.sql("""
SELECT count(*) AS pedidos, sum(CAST(flag_entregue_valido AS INT)) AS entregues_validos,
       sum(CAST(flag_atraso AS INT)) AS atrasados,
       round(100 * avg(CAST(flag_atraso AS INT)), 2) AS pct_atraso
FROM gold_fato_pedido""").show()